# Multimodal Text RAG Overview

### End-to-End Recipe Information Extraction and Hybrid Retrieval System

---

## Objective

This notebook demonstrates the complete workflow for building a Multimodal Text Retrieval-Augmented Generation (Text RAG) system using Bengali cooking videos.

The pipeline extracts information from multiple modalities—including video frames, audio, on-screen text (OCR), object detection, and visual descriptions—and converts them into structured textual representations. These representations are then transformed into searchable chunks, embedded into a vector database, indexed using BM25, and retrieved through a hybrid search mechanism.

The notebook walks through every stage of the pipeline, from raw video processing to hybrid retrieval, with explanations, code, and intermediate outputs.



# Step 1: Video Loading & Master JSON Creation

### Overview
Loads all cooking videos, extracts basic video metadata, and creates a Master JSON for each video. This Master JSON serves as the central data structure that is updated throughout the entire Multimodal Text RAG pipeline.

### Modules Used

- **OpenCV (cv2):** Opens video files and extracts metadata such as FPS, resolution, duration, and total frames.
- **json:** Creates and stores the Master JSON files.
- **pathlib.Path:** Handles file and directory operations in a platform-independent manner.

### Processing

- Reads all `.mp4` videos from the dataset.
- Extracts video metadata.
- Initializes a Master JSON containing placeholders for recipe information, audio, shots, frames, object detection, and visual descriptions.
- Saves one Master JSON file for each input video.

### Output

- Master JSON file for every video containing video metadata and empty sections for the remaining preprocessing stages.

In [1]:
# Import required libraries

import cv2                          # Read video properties
import json                         # Create JSON files
from pathlib import Path            # Handle file and folder paths


class VideoLoader:
    """Loads videos and creates one Master JSON for each video."""

    def __init__(self, video_directory):

        # Folder containing input videos
        self.video_directory = Path(video_directory)

        # Folder where Master JSON files will be stored
        self.json_directory = Path("data/json")

        # Create folder if it doesn't exist
        self.json_directory.mkdir(parents=True, exist_ok=True)

    def get_video_list(self):

        # Return all MP4 videos
        return sorted(self.video_directory.glob("*.mp4"))

    def load_video(self, video_path):

        # Open the video
        cap = cv2.VideoCapture(str(video_path))

        # Stop if video cannot be opened
        if not cap.isOpened():
            raise FileNotFoundError(f"Cannot open {video_path}")

        return cap

    def get_video_metadata(self, cap, video_path):

        # Read video properties
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        # Calculate duration
        duration = round(total_frames / fps, 2) if fps else 0

        # Calculate file size
        file_size = round(video_path.stat().st_size / (1024 * 1024), 2)

        # Store video metadata
        video_info = {

            "video_id": video_path.stem,

            "video_name": video_path.name,

            "title": video_path.stem.replace("_", " ").title(),

            "category": "Cooking",

            "language": "Bengali",

            "creator": "Debisha",

            "duration": duration,

            "fps": fps,

            "width": width,

            "height": height,

            "total_frames": total_frames,

            "file_size": f"{file_size} MB",

            "video_path": str(video_path)

        }

        return video_info

    def create_master_json(self, video_info):

        # Create the Master JSON structure
        master_json = {

            "video_info": video_info,

            "recipe_information": {

                "dish_type": "",

                "ingredients": [],

                "estimated_steps": [],

                "cuisine": "",

                "tags": []

            },

            "audio_information": {

                "transcript_bn": "",

                "transcript_en": "",

                "timestamps": []

            },

            "shot_information": [],

            "frame_information": [],

            "object_detection": [],

            "visual_descriptions": []

        }

        # JSON filename
        output_file = self.json_directory / f"{video_info['video_id']}.json"

        # Save Master JSON
        with open(output_file, "w", encoding="utf-8") as file:

            json.dump(master_json, file, indent=4, ensure_ascii=False)

        print(f" Master JSON created -> {output_file.name}")

    def process_all_videos(self):

        # Read all videos
        videos = self.get_video_list()

        print(f"\nFound {len(videos)} videos.\n")

        # Process each video
        for index, video in enumerate(videos, start=1):

            print(f"[{index}/{len(videos)}] Processing {video.name}")

            # Open video
            cap = self.load_video(video)

            # Extract metadata
            video_info = self.get_video_metadata(cap, video)

            # Create Master JSON
            self.create_master_json(video_info)

            # Release memory
            cap.release()

        print("\nMaster JSON creation completed successfully.")


if __name__ == "__main__":

    # Create VideoLoader object
    loader = VideoLoader("data/videos")

    # Process all videos
    loader.process_all_videos()


Found 5 videos.

[1/5] Processing CARAMEL_CUSTARD.mp4
 Master JSON created -> CARAMEL_CUSTARD.json
[2/5] Processing DAHI_CHICKEN.mp4
 Master JSON created -> DAHI_CHICKEN.json
[3/5] Processing FRIED_RICE.mp4
 Master JSON created -> FRIED_RICE.json
[4/5] Processing MANGO_CHICKEN_ROAST.mp4
 Master JSON created -> MANGO_CHICKEN_ROAST.json
[5/5] Processing POMFRET_ROAST_FRY.mp4
 Master JSON created -> POMFRET_ROAST_FRY.json

Master JSON creation completed successfully.


# Step 2: Shot Detection

### Overview
Identifies scene transitions in each cooking video and segments the video into multiple shots. The detected shot boundaries are stored in the Master JSON for use in frame extraction and subsequent processing stages.

### Modules Used

- **PySceneDetect:** Detects scene changes based on visual content differences.
- **ContentDetector:** Identifies shot boundaries by comparing consecutive video frames.
- **json:** Updates the `shot_information` section of the Master JSON.
- **pathlib.Path:** Handles video and JSON file paths.

### Processing

- Reads each input video.
- Detects shot boundaries using `ContentDetector`.
- Calculates the start time, end time, and duration of every shot.
- Updates the `shot_information` section in the corresponding Master JSON.

### Output

- Updated Master JSON containing shot boundaries with timestamps and shot durations for every video.

In [2]:

# Import Required Libraries
from pathlib import Path
import json

from scenedetect import detect
from scenedetect.detectors import ContentDetector


# Shot Detector
class ShotDetector:
    """Detects shots and updates the Master JSON."""

    def __init__(self, video_directory):

        # Folder containing videos
        self.video_directory = Path(video_directory)

        # Folder containing Master JSON files
        self.json_directory = Path("data/json")

   
    # Get Video List
    def get_video_list(self):

        return sorted(self.video_directory.glob("*.mp4"))

    
    
    # Detect Shots
 

    def detect_shots(self, video_path):

        scene_list = detect(

            str(video_path),

            ContentDetector(threshold=27.0)

        )

        shots = []

        for shot_number, scene in enumerate(scene_list, start=1):

            start_time = round(scene[0].seconds, 2)

            end_time = round(scene[1].seconds, 2)

            shots.append({

                "shot_id": f"shot_{shot_number}",

                "start_time": start_time,

                "end_time": end_time,

                "duration": round(end_time - start_time, 2),

                "representative_frame": ""

            })

        return shots

    
    # Update Master JSON


    def update_master_json(self, video_name, shots):

        json_file = self.json_directory / f"{Path(video_name).stem}.json"

        if not json_file.exists():

            print(f"JSON not found: {json_file}")

            return

        with open(json_file, "r", encoding="utf-8") as file:

            master_json = json.load(file)

        master_json["shot_information"] = shots

        with open(json_file, "w", encoding="utf-8") as file:

            json.dump(

                master_json,

                file,

                indent=4,

                ensure_ascii=False

            )

        print(f" Updated -> {json_file.name}")

    # Process All Videos
   

    def process_all_videos(self):

        videos = self.get_video_list()

        print(f"\nFound {len(videos)} videos.\n")

        for index, video in enumerate(videos, start=1):

            print(f"[{index}/{len(videos)}] Processing {video.name}")

            shots = self.detect_shots(video)

            print(f"Detected {len(shots)} shots")

            self.update_master_json(video.name, shots)

        print("\n Shot Detection Completed")


# Main


if __name__ == "__main__":

    detector = ShotDetector("data/videos")

    detector.process_all_videos()


Found 5 videos.

[1/5] Processing CARAMEL_CUSTARD.mp4
Detected 62 shots
 Updated -> CARAMEL_CUSTARD.json
[2/5] Processing DAHI_CHICKEN.mp4
Detected 41 shots
 Updated -> DAHI_CHICKEN.json
[3/5] Processing FRIED_RICE.mp4
Detected 33 shots
 Updated -> FRIED_RICE.json
[4/5] Processing MANGO_CHICKEN_ROAST.mp4
Detected 50 shots
 Updated -> MANGO_CHICKEN_ROAST.json
[5/5] Processing POMFRET_ROAST_FRY.mp4
Detected 55 shots
 Updated -> POMFRET_ROAST_FRY.json

 Shot Detection Completed


# Step 3: Representative Frame Extraction

### Overview
Extracts one representative frame from each detected shot by selecting the middle frame of the shot. The extracted frames are saved locally, and their metadata is added to the Master JSON for further OCR, object detection, and visual description generation.

### Modules Used

- **OpenCV (cv2):** Reads videos and extracts representative frames.
- **json:** Updates the `frame_information` and `shot_information` sections of the Master JSON.
- **pathlib.Path:** Handles file and directory operations.

### Processing

- Reads shot information from the Master JSON.
- Calculates the middle timestamp of each shot.
- Extracts and saves one representative frame for every shot.
- Updates the representative frame path and frame metadata in the Master JSON.

### Output

- Representative frames saved in `data/frames/`.
- Updated Master JSON containing frame paths, timestamps, and placeholders for OCR results.

In [5]:

# Import Required Libraries


import cv2
import json
from pathlib import Path


# Frame Extractor


class FrameExtractor:
    """
    Extract one representative frame from every detected shot
    and update the Master JSON.
    """

    def __init__(self, video_directory):

        # Folder containing videos
        self.video_directory = Path(video_directory)

        # Folder containing Master JSON files
        self.json_directory = Path("data/json")

        # Folder to save extracted frames
        self.frame_directory = Path("data/frames")

        # Create folder if it doesn't exist
        self.frame_directory.mkdir(
            parents=True,
            exist_ok=True
        )

    # Get Video List


    def get_video_list(self):

        return sorted(

            self.video_directory.glob("*.mp4")

        )

  
    # Extract Frames
    

    def extract_frames(self, video_path):

        print(f"\nProcessing {video_path.name}")

     
        # Open Master JSON
     

        json_file = self.json_directory / f"{video_path.stem}.json"

        with open(

            json_file,

            "r",

            encoding="utf-8"

        ) as file:

            master_json = json.load(file)

        # Read shot information
        shots = master_json.get(

            "shot_information",

            []

        )

        if len(shots) == 0:

            print("No shots found.")

            return

        
        # Open Video
       

        cap = cv2.VideoCapture(

            str(video_path)

        )

        fps = cap.get(

            cv2.CAP_PROP_FPS

        )

       
        # Folder for this video's frames
       

        video_frame_folder = self.frame_directory / video_path.stem

        video_frame_folder.mkdir(

            parents=True,

            exist_ok=True

        )

        # Store frame metadata

        frame_information = []

      
        # Process Every Shot
     

        for shot in shots:

            start_time = shot["start_time"]

            end_time = shot["end_time"]

            middle_time = (start_time + end_time) / 2

            frame_number = int(

                middle_time * fps

            )

            cap.set(

                cv2.CAP_PROP_POS_FRAMES,

                frame_number

            )

            success, frame = cap.read()

            if not success:

                continue

            
            # Save Representative Frame
           

            image_name = f"{shot['shot_id']}.jpg"

            image_path = video_frame_folder / image_name

            cv2.imwrite(

                str(image_path),

                frame

            )
            # Update representative frame inside shot_information
            

            shot["representative_frame"] = str(image_path)

           
            # Store Frame Information
           

            frame_information.append({

                "frame_id": f"{video_path.stem}_{shot['shot_id']}_f1",

                "shot_id": shot["shot_id"],

                "timestamp": round(middle_time, 2),

                "frame_path": str(image_path),

                "ocr_text": ""

            })

            print(f" Saved {image_name}")

        
        # Update Master JSON
      

        master_json["shot_information"] = shots

        master_json["frame_information"] = frame_information

     
        # Save Updated JSON
    

        with open(

            json_file,

            "w",

            encoding="utf-8"

        ) as file:

            json.dump(

                master_json,

                file,

                indent=4,

                ensure_ascii=False

            )

        cap.release()

        print(f" Updated -> {json_file.name}")

    
    # Process All Videos


    def process_all_videos(self):

        videos = self.get_video_list()

        print(f"\nFound {len(videos)} videos.\n")

        for index, video in enumerate(videos, start=1):

            print(f"[{index}/{len(videos)}] Processing {video.name}")

            self.extract_frames(video)

        print("\n Frame Extraction Completed Successfully")



# Main


if __name__ == "__main__":

    extractor = FrameExtractor("data/videos")

    extractor.process_all_videos()


Found 5 videos.

[1/5] Processing CARAMEL_CUSTARD.mp4

Processing CARAMEL_CUSTARD.mp4
 Saved shot_1.jpg
 Saved shot_2.jpg
 Saved shot_3.jpg
 Saved shot_4.jpg
 Saved shot_5.jpg
 Saved shot_6.jpg
 Saved shot_7.jpg
 Saved shot_8.jpg
 Saved shot_9.jpg
 Saved shot_10.jpg
 Saved shot_11.jpg
 Saved shot_12.jpg
 Saved shot_13.jpg
 Saved shot_14.jpg
 Saved shot_15.jpg
 Saved shot_16.jpg
 Saved shot_17.jpg
 Saved shot_18.jpg
 Saved shot_19.jpg
 Saved shot_20.jpg
 Saved shot_21.jpg
 Saved shot_22.jpg
 Saved shot_23.jpg
 Saved shot_24.jpg
 Saved shot_25.jpg
 Saved shot_26.jpg
 Saved shot_27.jpg
 Saved shot_28.jpg
 Saved shot_29.jpg
 Saved shot_30.jpg
 Saved shot_31.jpg
 Saved shot_32.jpg
 Saved shot_33.jpg
 Saved shot_34.jpg
 Saved shot_35.jpg
 Saved shot_36.jpg
 Saved shot_37.jpg
 Saved shot_38.jpg
 Saved shot_39.jpg
 Saved shot_40.jpg
 Saved shot_41.jpg
 Saved shot_42.jpg
 Saved shot_43.jpg
 Saved shot_44.jpg
 Saved shot_45.jpg
 Saved shot_46.jpg
 Saved shot_47.jpg
 Saved shot_48.jpg
 Saved sho

# Step 4: OCR Extraction

### Overview
Extracts textual information present in the representative frames using Optical Character Recognition (OCR). The extracted text is stored in the Master JSON and later contributes to chunk generation and retrieval.

### Modules Used

- **EasyOCR:** Detects and recognizes text from image frames.
- **json:** Updates the `ocr_text` field in the `frame_information` section of the Master JSON.
- **pathlib.Path:** Handles image and JSON file paths.

### Processing

- Reads all representative frames.
- Detects and extracts visible text from each frame.
- Associates the extracted text with its corresponding frame.
- Updates the Master JSON with OCR results.

### Output

- Updated Master JSON containing OCR text for each representative frame.

In [ ]:

# Import Required Libraries
import json
from pathlib import Path
import easyocr


#OCR Extractor
class OCRExtractor:
    """
    Reads all representative frames and extracts Bengali + English text
    using EasyOCR. Updates only the 'ocr_text' field inside
    frame_information.
    """

    def __init__(self):

        # Master JSON folder
        self.json_directory = Path("data/json")

        print("Loading EasyOCR...")

        # Bengali + English OCR
        self.reader = easyocr.Reader(
            ['bn', 'en'],
            gpu=False
        )

        print(" OCR Model Loaded")

    
    def get_json_files(self):

        return sorted(self.json_directory.glob("*.json"))

   
    def process_json(self, json_path):

        print(f"\nProcessing {json_path.name}")

        with open(json_path, "r", encoding="utf-8") as file:

            master_json = json.load(file)

        frame_information = master_json.get("frame_information", [])

        if len(frame_information) == 0:

            print("No frame information found.")
            return

       

        for index, frame in enumerate(frame_information, start=1):

            image_path = Path(frame["frame_path"])

            print(
                f"Frame {index}/{len(frame_information)} : {image_path.name}"
            )

            if not image_path.exists():

                print(f"Missing frame : {image_path}")

                frame["ocr_text"] = ""

                continue

            try:

                result = self.reader.readtext(
                    str(image_path),
                    detail=0,
                    paragraph=True
                )

                extracted_text = " ".join(result).strip()

                frame["ocr_text"] = extracted_text

            except Exception as e:

                print(e)

                frame["ocr_text"] = ""

       
        master_json["frame_information"] = frame_information

        with open(json_path, "w", encoding="utf-8") as file:

            json.dump(
                master_json,
                file,
                indent=4,
                ensure_ascii=False
            )

        print(f" Updated -> {json_path.name}")

    

    def process_all_json(self):

        json_files = self.get_json_files()

        print(f"\nFound {len(json_files)} JSON files.\n")

        for json_file in json_files:

            self.process_json(json_file)

        print("\n OCR Extraction Completed Successfully")



# Main


if __name__ == "__main__":

    extractor = OCRExtractor()

    extractor.process_all_json()

Using CPU. Note: This module is much faster with a GPU.


Loading EasyOCR...


/home/debisha/video-graph-rag/.venv/lib/python3.12/site-packages/torch/ao/nn/quantized/dynamic/modules/rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/quantized/Quantizer.cpp:111.)
  w_ih = torch.quantize_per_tensor(


 OCR Model Loaded

Found 5 JSON files.


Processing CARAMEL_CUSTARD.json
Frame 1/62 : shot_1.jpg


/home/debisha/video-graph-rag/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

Frame 48/55 : shot_48.jpg
Frame 49/55 : shot_49.jpg


/home/debisha/video-graph-rag/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Frame 50/55 : shot_50.jpg
Frame 51/55 : shot_51.jpg
Frame 52/55 : shot_52.jpg
Frame 53/55 : shot_53.jpg
Frame 54/55 : shot_54.jpg


/home/debisha/video-graph-rag/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Frame 55/55 : shot_55.jpg
 Updated -> POMFRET_ROAST_FRY.json

 OCR Extraction Completed Successfully


# Step 5: Audio Extraction

### Overview

Extracts the audio stream from each input video and converts it into a standard WAV format. The extracted audio is used in the subsequent speech-to-text transcription stage.

### Modules Used

- **subprocess:** Executes FFmpeg commands from Python.
- **FFmpeg:** Extracts audio from videos and converts it to 16 kHz mono WAV format.
- **pathlib.Path:** Handles file and directory operations.

### Processing

- Reads all input video files.
- Extracts the audio stream from each video using FFmpeg.
- Converts the audio to **16 kHz**, **mono-channel**, **PCM WAV** format.
- Saves the extracted audio in the designated output directory.

### Output

- One `.wav` audio file for each input video stored in `data/audio/`.

In [8]:
# Import required libraries

import subprocess                   # Run FFmpeg commands
from pathlib import Path            # Handle file and folder paths


class AudioExtractor:
    """Extracts audio from every video using FFmpeg."""

    def __init__(self, video_directory):

        # Folder containing videos
        self.video_directory = Path(video_directory)

        # Folder to save extracted audio
        self.audio_directory = Path("data/audio")

        # Create audio folder if it doesn't exist
        self.audio_directory.mkdir(parents=True, exist_ok=True)

    def get_video_list(self):

        # Return all MP4 videos
        return sorted(self.video_directory.glob("*.mp4"))

    def extract_audio(self, video_path):

        print(f"\nProcessing {video_path.name}")

        # Create output audio filename
        audio_file = self.audio_directory / f"{video_path.stem}.wav"

        # FFmpeg command
        command = [

            "ffmpeg",

            "-i", str(video_path),

            "-vn",

            "-acodec", "pcm_s16le",

            "-ar", "16000",

            "-ac", "1",

            "-y",

            str(audio_file)

        ]

        # Execute FFmpeg command
        subprocess.run(

            command,

            stdout=subprocess.DEVNULL,

            stderr=subprocess.DEVNULL

        )

        print(f"✓ Audio saved -> {audio_file.name}")

    def process_all_videos(self):

        # Read all videos
        videos = self.get_video_list()

        print(f"\nFound {len(videos)} videos.\n")

        # Process every video
        for index, video in enumerate(videos, start=1):

            print(f"[{index}/{len(videos)}]")

            self.extract_audio(video)

        print("\nAudio extraction completed successfully.")


if __name__ == "__main__":

    # Create AudioExtractor object
    extractor = AudioExtractor("data/videos")

    # Process all videos
    extractor.process_all_videos()


Found 5 videos.

[1/5]

Processing CARAMEL_CUSTARD.mp4
✓ Audio saved -> CARAMEL_CUSTARD.wav
[2/5]

Processing DAHI_CHICKEN.mp4
✓ Audio saved -> DAHI_CHICKEN.wav
[3/5]

Processing FRIED_RICE.mp4
✓ Audio saved -> FRIED_RICE.wav
[4/5]

Processing MANGO_CHICKEN_ROAST.mp4
✓ Audio saved -> MANGO_CHICKEN_ROAST.wav
[5/5]

Processing POMFRET_ROAST_FRY.mp4
✓ Audio saved -> POMFRET_ROAST_FRY.wav

Audio extraction completed successfully.


# Step 6: Audio Transcription & Translation

### Overview

Transcribes the extracted Bengali audio into text using the **Sarvam AI Speech-to-Text API**. The generated transcript is then translated into English, and both transcripts along with timestamp information are stored in the Master JSON.

### Modules Used

- **Sarvam AI:** Performs Bengali speech-to-text transcription with timestamp generation.
- **Translator:** Converts the Bengali transcript into English.
- **dotenv:** Loads the Sarvam API key securely from the `.env` file.
- **json:** Updates the `audio_information` section of the Master JSON.
- **pathlib.Path:** Handles audio, transcript, and JSON file paths.

### Processing

- Reads each extracted audio file.
- Uploads the audio to the Sarvam AI Speech-to-Text service.
- Generates the Bengali transcript with timestamps.
- Translates the transcript into English.
- Updates the Master JSON with the Bengali transcript, English transcript, and timestamp information.

### Output

- Updated Master JSON containing:
  - Bengali transcript
  - English transcript
  - Timestamped transcript segments

In [ ]:

# Import Required Libraries


import os
import json
from pathlib import Path

from dotenv import load_dotenv
from sarvamai import SarvamAI

# Import Required Libraries


from deep_translator import GoogleTranslator



# Translator Class


class Translator:
    """
    Translates Bengali text into English using Google Translator.
    """

    def __init__(self):

        # Initialize translator
        self.translator = GoogleTranslator(
            source="bn",
            target="en"
        )

    

    def translate(self, bengali_text):
        """
        Translate Bengali text to English.
        """

        if bengali_text is None:
            return ""

        bengali_text = bengali_text.strip()

        if bengali_text == "":
            return ""

        try:

            english_text = self.translator.translate(
                bengali_text
            )

            return english_text

        except Exception as e:

            print(f"Translation Error: {e}")

            return ""


# Audio Transcriber


class AudioTranscriber:
    """
    Transcribes every extracted audio file using Sarvam AI
    and updates the Master JSON.
    """

    def __init__(self, audio_directory):

      
        # Load Environment Variables
       

        load_dotenv()

        api_key = os.getenv("SARVAM_API_KEY")

        if not api_key:

            raise ValueError(
                " SARVAM_API_KEY not found in .env"
            )

        # Sarvam Client
      
        self.client = SarvamAI(

            api_subscription_key=api_key

        )

      
        # Translator
        

        self.translator = Translator()

        
        # Directories
        

        self.audio_directory = Path(audio_directory)

        self.json_directory = Path("data/json")

        self.output_directory = Path("data/transcripts")

        self.output_directory.mkdir(

            parents=True,

            exist_ok=True

        )

    
    # Get Audio Files
    
    def get_audio_files(self):

        return sorted(

            self.audio_directory.glob("*.wav")

        )

    
    # Process One Audio File
    
    def process_audio(self, audio_file):

        print(f"\nProcessing {audio_file.name}")

        
        # Find Corresponding Master JSON
        
        master_json_path = (

            self.json_directory /

            f"{audio_file.stem}.json"

        )

        if not master_json_path.exists():

            print(

                f" Master JSON not found for {audio_file.stem}"

            )

            return

        
        # Load Master JSON
        

        with open(

            master_json_path,

            "r",

            encoding="utf-8"

        ) as file:

            master_json = json.load(file)
               
        # Create Speech-to-Text Job
        
        print("Creating Sarvam Job...")

        try:

            job = self.client.speech_to_text_job.create_job(

                model="saaras:v3",

                mode="transcribe",

                language_code="bn-IN",

                with_diarization=False

            )

            print(" Job Created")

        except Exception as e:

            print(f" Failed to create job:\n{e}")

            return

       
        # Upload Audio File
        
        print("Uploading Audio...")

        try:

            job.upload_files(

                file_paths=[

                    str(audio_file)

                ]

            )

            print(" Audio Uploaded")

        except Exception as e:

            print(f" Upload Failed:\n{e}")

            return

        
        # Start Job
       
        print("Starting Transcription...")

        try:

            job.start()

            print(" Job Started")

        except Exception as e:

            print(f" Failed to start job:\n{e}")

            return

        
        # Wait Until Completed
        
        print("Waiting for Sarvam AI...")

        try:

            job.wait_until_complete()

            print(" Transcription Completed")

        except Exception as e:

            print(f" Job Failed:\n{e}")

            return

        
        # Download Output Files
       
        print("Downloading Transcript...")

        try:

            job.download_outputs(

                output_dir=str(

                    self.output_directory

                )

            )

            print(" Transcript Downloaded")

        except Exception as e:

            print(f" Download Failed:\n{e}")

            return

       
        # Locate Downloaded JSON
       
        json_files = sorted(

            self.output_directory.glob("*.json"),

            key=lambda file: file.stat().st_mtime,

            reverse=True

        )

        if len(json_files) == 0:

            print(" No transcript JSON found.")

            return

        transcript_json = json_files[0]

        print(f"Using Transcript: {transcript_json.name}")

       
        # Load Transcript JSON
        

        with open(

            transcript_json,

            "r",

            encoding="utf-8"

        ) as file:

            result = json.load(file)
               
        # Bengali Transcript
       
        transcript_bn = result.get("transcript", "").strip()

        if transcript_bn == "":

            print(" Empty transcript received.")

            return

        print(" Bengali Transcript Loaded")

       
        # English Translation
       
        print("Translating to English...")

        try:

            transcript_en = self.translator.translate(

                transcript_bn

            )

            print(" English Translation Completed")

        except Exception as e:

            print(f"Translation Failed: {e}")

            transcript_en = ""

       
        # Extract Timestamps
        
        timestamps = []

        try:

            timestamp_data = result.get("timestamps", {})

            chunks = timestamp_data.get("chunks", [])

            starts = timestamp_data.get("start_time_seconds", [])

            ends = timestamp_data.get("end_time_seconds", [])

            if len(chunks) == len(starts) == len(ends):

                for text, start, end in zip(

                    chunks,

                    starts,

                    ends

                ):

                    timestamps.append({

                        "start": round(float(start), 2),

                        "end": round(float(end), 2),

                        "text": text.strip()

                    })

                print(f" {len(timestamps)} timestamps extracted")

            else:

                print(" Timestamp lengths do not match.")

        except Exception as e:

            print(f" Timestamp Extraction Failed: {e}")

            timestamps = []

        
        # Update Master JSON
       
        master_json["audio_information"] = {

            "transcript_bn": transcript_bn,

            "transcript_en": transcript_en,

            "timestamps": timestamps

        }

       
        # Save Master JSON
       
        with open(

            master_json_path,

            "w",

            encoding="utf-8"

        ) as file:

            json.dump(

                master_json,

                file,

                indent=4,

                ensure_ascii=False

            )

        print(f" Updated -> {master_json_path.name}")
           
    # Process All Audio Files
    

    def process_all_audio(self):

        audio_files = self.get_audio_files()

        print(f"\nFound {len(audio_files)} audio files.\n")

        if len(audio_files) == 0:

            print(" No audio files found.")

            return

        for index, audio_file in enumerate(audio_files, start=1):

            print("=" * 70)

            print(f"[{index}/{len(audio_files)}] {audio_file.name}")

            print("=" * 70)

            try:

                self.process_audio(audio_file)

            except Exception as e:

                print(f" Error processing {audio_file.name}")

                print(e)

        print("\n" + "=" * 70)

        print(" Audio Transcription Completed Successfully")

        print("=" * 70)



# Main

if __name__ == "__main__":

    transcriber = AudioTranscriber(

        "data/audio"

    )

    transcriber.process_all_audio()


Found 5 audio files.

[1/5] CARAMEL_CUSTARD.wav

Processing CARAMEL_CUSTARD.wav
Creating Sarvam Job...
 Job Created
Uploading Audio...
 Audio Uploaded
Starting Transcription...
 Job Started
Waiting for Sarvam AI...
 Transcription Completed
 Transcript Downloaded
Using Transcript: CARAMEL_CUSTARD.wav.json
 Bengali Transcript Loaded
Translating to English...
 English Translation Completed
 Timestamp Extraction Failed: 'NoneType' object has no attribute 'get'
 Updated -> CARAMEL_CUSTARD.json
[2/5] DAHI_CHICKEN.wav

Processing DAHI_CHICKEN.wav
Creating Sarvam Job...
 Job Created
Uploading Audio...
 Audio Uploaded
Starting Transcription...
 Job Started
Waiting for Sarvam AI...
 Transcription Completed
 Transcript Downloaded
Using Transcript: DAHI_CHICKEN.wav.json
 Bengali Transcript Loaded
Translating to English...
 English Translation Completed
 Timestamp Extraction Failed: 'NoneType' object has no attribute 'get'
 Updated -> DAHI_CHICKEN.json
[3/5] FRIED_RICE.wav

Processing FRIED_RICE.w

## Step 7: Audio–Shot Alignment

### Input
- Master JSON files containing:
  - Shot information
  - Bengali and English transcripts
- Bengali to English Translator

### Processing
- Splits the Bengali transcript into sentence-level segments.
- Translates each Bengali sentence into English.
- Aligns transcript segments with detected video shots using shot timings.
- Updates every shot with its corresponding Bengali and English audio text.
- Merges consecutive shots containing the same instruction into **semantic cooking steps**.

### Output
- Updates the `audio_information.timestamps` field in the Master JSON.
- Adds `audio_text_bn` and `audio_text_en` to each shot.
- Creates `semantic_shot_information` for higher-level cooking steps.
- Saves the updated Master JSON for every video.

In [1]:

# Creates timestamps from Sarvam transcript using shot timings,
# aligns transcript to detected shots,
# and generates semantic shot information.

import json
from pathlib import Path

# Import Required Libraries

from deep_translator import GoogleTranslator



# Translator Class

class Translator:
    """
    Translates Bengali text into English using Google Translator.
    """

    def __init__(self):

        # Initialize translator
        self.translator = GoogleTranslator(
            source="bn",
            target="en"
        )

    

    def translate(self, bengali_text):
        """
        Translate Bengali text to English.
        """

        if bengali_text is None:
            return ""

        bengali_text = bengali_text.strip()

        if bengali_text == "":
            return ""

        try:

            english_text = self.translator.translate(
                bengali_text
            )

            return english_text

        except Exception as e:

            print(f"Translation Error: {e}")

            return ""

# Audio Shot Aligner

class AudioShotAligner:

    def __init__(self):

        # Master JSON folder
        self.json_directory = Path("data/json")

        # Bengali -> English translator
        self.translator = Translator()

   
    # Get all Master JSON files
   
    def get_json_files(self):

        return sorted(

            self.json_directory.glob("*.json")

        )

    
    # Merge consecutive semantic shots
    

    def merge_semantic_shots(self, shots):

        semantic_shots = []

        current = None

        step_id = 1

        for shot in shots:

            text_bn = shot.get(
                "audio_text_bn",
                ""
            ).strip()

            text_en = shot.get(
                "audio_text_en",
                ""
            ).strip()

            # Ignore silent shots
            if text_bn == "":
                continue

            # First semantic step
            if current is None:

                current = {

                    "step_id": step_id,

                    "start_time": shot["start_time"],

                    "end_time": shot["end_time"],

                    "duration": shot["duration"],

                    "representative_frame":
                        shot["representative_frame"],

                    "audio_text_bn": text_bn,

                    "audio_text_en": text_en

                }

                continue

            # Same instruction -> merge
            if current["audio_text_bn"] == text_bn:

                current["end_time"] = shot["end_time"]

                current["duration"] = round(

                    current["end_time"]
                    -
                    current["start_time"],

                    2

                )

            else:

                semantic_shots.append(current)

                step_id += 1

                current = {

                    "step_id": step_id,

                    "start_time": shot["start_time"],

                    "end_time": shot["end_time"],

                    "duration": shot["duration"],

                    "representative_frame":
                        shot["representative_frame"],

                    "audio_text_bn": text_bn,

                    "audio_text_en": text_en

                }

        if current is not None:

            semantic_shots.append(current)

        return semantic_shots
       
    # Process One Master JSON
   

    def process_json(self, json_path):

        print(f"\nProcessing : {json_path.name}")

        
        # Load Master JSON
        

        with open(

            json_path,

            "r",

            encoding="utf-8"

        ) as file:

            master_json = json.load(file)

       
        # Read Audio Information
        
        audio_information = master_json.get(

            "audio_information",

            {}

        )

        transcript_bn = audio_information.get(

            "transcript_bn",

            ""

        ).strip()

        transcript_en = audio_information.get(

            "transcript_en",

            ""

        ).strip()

        if transcript_bn == "":

            print(" No transcript found.")

            return

       
        # Read Shot Information
        
        shots = master_json.get(

            "shot_information",

            []

        )

        if len(shots) == 0:

            print(" No shot information found.")

            return

        print(f"Total Shots : {len(shots)}")

        
        # Split Bengali Transcript into Sentences
        
        sentences_bn = [

            sentence.strip()

            for sentence in transcript_bn.split("।")

            if sentence.strip()

        ]

        # If transcript has no Bengali punctuation,
        # keep the whole transcript as one sentence.

        if len(sentences_bn) == 0:

            sentences_bn = [

                transcript_bn

            ]

        print(f"Transcript Sentences : {len(sentences_bn)}")

        
        # Translate Each Sentence
        
        sentences_en = []

        for sentence in sentences_bn:

            try:

                translated = self.translator.translate(

                    sentence

                )

            except Exception:

                translated = ""

            sentences_en.append(

                translated

            )

        
        # Initialize Shot Fields
        

        for shot in shots:

            shot["audio_text"] = ""

            shot["audio_text_bn"] = ""

            shot["audio_text_en"] = ""
               
        # Generate Timestamp Segments from Shot Timings
        

        timestamps = []

        total_shots = len(shots)

        total_sentences = len(sentences_bn)

        # Number of shots assigned to each sentence
        shots_per_sentence = max(

            1,

            round(total_shots / total_sentences)

        )

        sentence_index = 0

        for shot_index, shot in enumerate(shots):

            # Move to next sentence after enough shots
            if (

                shot_index != 0

                and

                shot_index % shots_per_sentence == 0

                and

                sentence_index < total_sentences - 1

            ):

                sentence_index += 1

            start_time = shot["start_time"]

            end_time = shot["end_time"]

            text_bn = sentences_bn[sentence_index]

            text_en = sentences_en[sentence_index]

           
            # Store timestamp
            

            timestamps.append(

                {

                    "start": start_time,

                    "end": end_time,

                    "text": text_bn

                }

            )

           
            # Update Shot Information
           

            shot["audio_text"] = text_bn

            shot["audio_text_bn"] = text_bn

            shot["audio_text_en"] = text_en

        print(

            f" Generated {len(timestamps)} timestamp segments"

        )

        
        # Save timestamps back into audio_information
       
        audio_information["timestamps"] = timestamps

        master_json["audio_information"] = audio_information
                
        # Generate Semantic Shot Information
       
        print("\nGenerating Semantic Shot Information...")

        semantic_shots = self.merge_semantic_shots(

            shots

        )

        print(

            f" Generated {len(semantic_shots)} semantic steps"

        )

       
        # Update Master JSON
       
        master_json["audio_information"] = audio_information

        master_json["shot_information"] = shots

        master_json["semantic_shot_information"] = semantic_shots

        print(" Master JSON Updated")
               
        # Save Updated Master JSON
       
        with open(

            json_path,

            "w",

            encoding="utf-8"

        ) as file:

            json.dump(

                master_json,

                file,

                indent=4,

                ensure_ascii=False

            )

        print(f" Updated -> {json_path.name}")

   
    # Process All JSON Files
    
    def process_all_json(self):

        json_files = self.get_json_files()

       
        print("Audio Shot Alignment Started")
       

        print(f"\nFound {len(json_files)} JSON files.\n")

        for json_file in json_files:

            self.process_json(

                json_file

            )

       
        print(" Audio Shot Alignment Completed")
       



# Main

if __name__ == "__main__":

    aligner = AudioShotAligner()

    aligner.process_all_json()

Audio Shot Alignment Started

Found 5 JSON files.


Processing : CARAMEL_CUSTARD.json
Total Shots : 62
Transcript Sentences : 9
 Generated 62 timestamp segments

Generating Semantic Shot Information...
 Generated 9 semantic steps
 Master JSON Updated
 Updated -> CARAMEL_CUSTARD.json

Processing : DAHI_CHICKEN.json
Total Shots : 41
Transcript Sentences : 7
 Generated 41 timestamp segments

Generating Semantic Shot Information...
 Generated 7 semantic steps
 Master JSON Updated
 Updated -> DAHI_CHICKEN.json

Processing : FRIED_RICE.json
Total Shots : 33
Transcript Sentences : 7
 Generated 33 timestamp segments

Generating Semantic Shot Information...
 Generated 7 semantic steps
 Master JSON Updated
 Updated -> FRIED_RICE.json

Processing : MANGO_CHICKEN_ROAST.json
Total Shots : 50
Transcript Sentences : 9
 Generated 50 timestamp segments

Generating Semantic Shot Information...
 Generated 9 semantic steps
 Master JSON Updated
 Updated -> MANGO_CHICKEN_ROAST.json

Processing : POMFRET_ROA

## Step 8: Object Detection

### Input
- Master JSON files containing representative frame information.
- Extracted representative frames from each detected shot.
- Pre-trained YOLOv8 Nano (`yolov8n`) object detection model.

### Processing
- Loads the representative frame for each shot.
- Uses the YOLOv8 model to detect visible objects in the frame.
- Extracts the object class, confidence score, and bounding box coordinates.
- Stores all detected objects for each frame.

### Output
- Updates the `object_detection` field in the Master JSON.
- Stores detected object names, confidence scores, and bounding box coordinates for every representative frame.
- Saves the updated Master JSON for each video.

In [2]:

# Import Required Libraries

import json
from pathlib import Path

from deep_translator import GoogleTranslator



# Translator Class


class Translator:
    """
    Translates Bengali text into English.
    """

    def __init__(self):

        self.translator = GoogleTranslator(

            source="bn",

            target="en"

        )

    def translate(

        self,

        bengali_text

    ):

        if bengali_text is None:

            return ""

        bengali_text = bengali_text.strip()

        if bengali_text == "":

            return ""

        try:

            return self.translator.translate(

                bengali_text

            )

        except Exception:

            return ""



# Audio Shot Aligner


class AudioShotAligner:

    def __init__(self):

        # Folder containing Master JSON files

        self.json_directory = Path(

            "data/json"

        )

        # Translator

        self.translator = Translator()


    
    # Get All Master JSON Files
    
    def get_json_files(

        self

    ):

        return sorted(

            self.json_directory.glob(

                "*.json"

            )

        )


   
    # Process One Master JSON
    
    def process_json(

        self,

        json_path

    ):

        print(

            f"\nProcessing : {json_path.name}"

        )

        # Load Master JSON

        with open(

            json_path,

            "r",

            encoding="utf-8"

        ) as file:

            master_json = json.load(

                file

            )

        # Read Audio Information

        audio_information = master_json.get(

            "audio_information",

            {}

        )

        # Read Shot Information

        shots = master_json.get(

            "shot_information",

            []

        )

        if len(shots) == 0:

            print(

                "No shot information found."

            )

            return

        print(

            f"Total Shots : {len(shots)}"

        )

        # Read Existing Speech Timestamps

        speech_timestamps = audio_information.get(

            "timestamps",

            []

        )

        if len(speech_timestamps) == 0:

            print(

                "No speech timestamps found."

            )

            return
                # Generate Timestamp Segments

        timestamps = []

        for segment in speech_timestamps:

            speech_start = segment.get(

                "start",

                0.0

            )

            speech_end = segment.get(

                "end",

                0.0

            )

            text_bn = segment.get(

                "text",

                ""

            ).strip()

            if text_bn == "":

                continue

            # Translate Bengali Segment

            text_en = self.translator.translate(

                text_bn

            )

            # Find All Overlapping Shots

            overlapping_shots = []

            for shot in shots:

                shot_start = shot["start_time"]

                shot_end = shot["end_time"]

                if (

                    shot_end >= speech_start

                    and

                    shot_start <= speech_end

                ):

                    overlapping_shots.append(

                        shot

                    )

            # Skip if no matching shot is found

            if len(overlapping_shots) == 0:

                continue

            # Create Timestamp Segment

            timestamps.append(

                {

                    "start": overlapping_shots[0]["start_time"],

                    "end": overlapping_shots[-1]["end_time"],

                    "text_bn": text_bn,

                    "text_en": text_en

                }

            )

        print(

            f"Generated {len(timestamps)} timestamp segments"

        )
                # Update Audio Information

        audio_information["timestamps"] = timestamps

        master_json["audio_information"] = audio_information


        # Save Updated Master JSON

        with open(

            json_path,

            "w",

            encoding="utf-8"

        ) as file:

            json.dump(

                master_json,

                file,

                indent=4,

                ensure_ascii=False

            )

        print(

            f"Updated -> {json_path.name}"

        )


    
    # Process All Master JSON Files
   
    def process_all_json(

        self

    ):

        json_files = self.get_json_files()

        print(

            "\nAudio Shot Alignment Started"

        )

        print(

            f"\nFound {len(json_files)} JSON files.\n"

        )

        for json_file in json_files:

            self.process_json(

                json_file

            )

        print(

            "\nAudio Shot Alignment Completed"

        )

# Main


if __name__ == "__main__":

    # Create Audio Shot Aligner object

    aligner = AudioShotAligner()

    # Process all Master JSON files

    aligner.process_all_json()


Audio Shot Alignment Started

Found 5 JSON files.


Processing : CARAMEL_CUSTARD.json
Total Shots : 62
Generated 62 timestamp segments
Updated -> CARAMEL_CUSTARD.json

Processing : DAHI_CHICKEN.json
Total Shots : 41
Generated 41 timestamp segments
Updated -> DAHI_CHICKEN.json

Processing : FRIED_RICE.json
Total Shots : 33
Generated 33 timestamp segments
Updated -> FRIED_RICE.json

Processing : MANGO_CHICKEN_ROAST.json
Total Shots : 50
Generated 50 timestamp segments
Updated -> MANGO_CHICKEN_ROAST.json

Processing : POMFRET_ROAST_FRY.json
Total Shots : 55
Generated 55 timestamp segments
Updated -> POMFRET_ROAST_FRY.json

Audio Shot Alignment Completed


## Step 9: Visual Description Generation

### Input
- Master JSON files containing frame information, object detection results, and shot-level transcripts.
- OCR text extracted from representative frames.
- Gemma 3 Large Language Model (LLM).

### Processing
- Combines OCR text, detected objects, and the corresponding transcript for each shot.
- Uses Gemma 3 to generate a concise visual description of the cooking scene.
- Focuses on ingredients, utensils, cooking actions, and food appearance.
- Stores one visual description for each representative frame.

### Output
- Updates the `visual_descriptions` field in the Master JSON.
- Stores a natural language description for every representative frame.
- Saves the updated Master JSON for each video.

In [3]:

# Import Required Libraries

import json
from pathlib import Path

import ollama



# Visual Description Generator

class VisualDescriptionGenerator:

    def __init__(self):

        # Master JSON folder
        self.json_directory = Path("data/json")

        print("Loading Gemma 3...")

        # Text Model
        self.model_name = "gemma3:latest"

        print(f" Model Loaded : {self.model_name}")


   
    # Get all Master JSON files
    
    def get_json_files(self):

        return sorted(

            self.json_directory.glob("*.json")

        )
            
    # Generate Description using OCR + Objects + Transcript
    
    def generate_description(

        self,

        ocr_text,

        detected_objects,

        transcript

    ):

       
        # Convert detected objects into text
       
        object_names = []

        for obj in detected_objects:

            object_names.append(

                obj["class_name"]

            )

        object_text = ", ".join(

            sorted(

                set(object_names)

            )

        )

        if object_text == "":

            object_text = "None"

        if ocr_text.strip() == "":

            ocr_text = "None"

        if transcript.strip() == "":

            transcript = "None"

       
        # Prompt
        
        prompt = f"""
You are generating a visual description for a cooking video.

Use the following information:

OCR Text:
{ocr_text}

Detected Objects:
{object_text}

Recipe Transcript:
{transcript}

Generate ONLY one concise description (1-2 sentences).

Focus on:
- ingredients
- utensils
- cooking action
- food appearance

Do not mention OCR, transcript or object detection.
Do not invent information not supported by the provided inputs.
Return only the description.
"""

       
        # Call Gemma 3 (TEXT ONLY)
        
        try:

            response = ollama.chat(

                model=self.model_name,

                messages=[

                    {

                        "role": "user",

                        "content": prompt

                    }

                ]

            )

            return response["message"]["content"].strip()

        except Exception as e:

            print(e)

            return ""
        
    # Process One Master JSON
    
    def process_json(self, json_path):

        print(f"\nProcessing {json_path.name}")

        
        # Load Master JSON
        
        with open(

            json_path,

            "r",

            encoding="utf-8"

        ) as file:

            master_json = json.load(file)

        
        # Read Frame Information
        

        frame_information = master_json.get(

            "frame_information",

            []

        )

        
        # Read Object Detection
       
        object_detection = master_json.get(

            "object_detection",

            []

        )

        
        # Read Shot Information
       
        shot_information = master_json.get(

            "shot_information",

            []

        )

        if len(frame_information) == 0:

            print("No frame information found.")

            return

        print(

            f"Total Frames : {len(frame_information)}"

        )

        
        # Build Object Lookup
        

        object_lookup = {

            item["frame_id"]: item["detected_objects"]

            for item in object_detection

        }

       
        # Build Shot Lookup
       
        shot_lookup = {

            shot["shot_id"]: shot

            for shot in shot_information

        }

        
        # Existing Visual Descriptions
        

        visual_descriptions = master_json.get(

            "visual_descriptions",

            []

        )

        completed_frames = {

            item["frame_id"]

            for item in visual_descriptions

        }

        print(

            f"Already Completed : {len(completed_frames)}"

        )

        
        # Process Every Representative Frame
        

        for frame in frame_information:

            frame_id = frame["frame_id"]

            shot_id = frame["shot_id"]

           
            # Skip Already Processed Frames
            

            if frame_id in completed_frames:

                print(

                    f"Skipping {frame_id}"

                )

                continue

           
            # OCR
            
            ocr_text = frame.get(

                "ocr_text",

                ""

            )

            
            # Objects
            
            detected_objects = object_lookup.get(

                frame_id,

                []

            )

           
            # Transcript
            

            transcript = ""

            if shot_id in shot_lookup:

                transcript = shot_lookup[

                    shot_id

                ].get(

                    "audio_text_en",

                    ""

                )

            print(

                f"\nGenerating Description -> {frame_id}"

            )

            description = self.generate_description(

                ocr_text,

                detected_objects,

                transcript

            )
                 
            # Save Generated Description
            
            if description != "":

                visual_descriptions.append(

                    {
                        "frame_id": frame_id,

                        "shot_id": shot_id,

                        "timestamp": frame.get(
                            "timestamp",
                            0
                        ),

                        "description": description
                    }

                )


                completed_frames.add(frame_id)


               
                # Save Immediately (Checkpoint)
                

                master_json["visual_descriptions"] = visual_descriptions


                with open(

                    json_path,

                    "w",

                    encoding="utf-8"

                ) as file:

                    json.dump(

                        master_json,

                        file,

                        indent=4,

                        ensure_ascii=False

                    )


                print(

                    f" Saved Description : {frame_id}"

                )


            else:

                print(

                    f" Failed Description : {frame_id}"

                )


        print(

            f"\nCompleted {json_path.name}"

        )

# Main Execution


if __name__ == "__main__":


    generator = VisualDescriptionGenerator()


    json_files = generator.get_json_files()


    print(
        f"\nTotal JSON Files Found : {len(json_files)}"
    )


    for json_file in json_files:

        generator.process_json(json_file)


    print(
        "\n================================="
    )

    print(
        "ALL VISUAL DESCRIPTIONS COMPLETED"
    )

    print(
        "================================="
    )

Loading Gemma 3...
 Model Loaded : gemma3:latest

Total JSON Files Found : 5

Processing CARAMEL_CUSTARD.json
Total Frames : 62
Already Completed : 62
Skipping CARAMEL_CUSTARD_shot_1_f1
Skipping CARAMEL_CUSTARD_shot_2_f1
Skipping CARAMEL_CUSTARD_shot_3_f1
Skipping CARAMEL_CUSTARD_shot_4_f1
Skipping CARAMEL_CUSTARD_shot_5_f1
Skipping CARAMEL_CUSTARD_shot_6_f1
Skipping CARAMEL_CUSTARD_shot_7_f1
Skipping CARAMEL_CUSTARD_shot_8_f1
Skipping CARAMEL_CUSTARD_shot_9_f1
Skipping CARAMEL_CUSTARD_shot_10_f1
Skipping CARAMEL_CUSTARD_shot_11_f1
Skipping CARAMEL_CUSTARD_shot_12_f1
Skipping CARAMEL_CUSTARD_shot_13_f1
Skipping CARAMEL_CUSTARD_shot_14_f1
Skipping CARAMEL_CUSTARD_shot_15_f1
Skipping CARAMEL_CUSTARD_shot_16_f1
Skipping CARAMEL_CUSTARD_shot_17_f1
Skipping CARAMEL_CUSTARD_shot_18_f1
Skipping CARAMEL_CUSTARD_shot_19_f1
Skipping CARAMEL_CUSTARD_shot_20_f1
Skipping CARAMEL_CUSTARD_shot_21_f1
Skipping CARAMEL_CUSTARD_shot_22_f1
Skipping CARAMEL_CUSTARD_shot_23_f1
Skipping CARAMEL_CUSTARD_shot_

## Step 10: Recipe Information Generation

### Input
- Master JSON files containing the English recipe transcript.
- Gemma 3 Large Language Model (LLM).

### Processing
- Uses the English transcript to understand the complete cooking procedure.
- Extracts structured recipe information using Gemma 3.
- Identifies the recipe name, ingredients, cooking steps, cuisine, and relevant tags.
- Organizes the extracted information into a structured JSON format.

### Output
- Updates the `recipe_information` field in the Master JSON.
- Stores the dish type, ingredients, estimated cooking steps, cuisine, and recipe tags.
- Saves the updated Master JSON for each video.

In [ ]:

# Import Required Libraries


import json
from pathlib import Path

import ollama



# Recipe Information Generator


class RecipeInformationGenerator:

    def __init__(self):

        # Master JSON Folder
        self.json_directory = Path("data/json")

        print("Loading Gemma 3 Model...")

        # Ollama Model
        self.model_name = "gemma3:latest"

        print(f" Model Loaded : {self.model_name}")


    
    # Get All Master JSON Files
    

    def get_json_files(self):

        return sorted(

            self.json_directory.glob("*.json")

        )
    
    # Generate Recipe Information using Gemma 3
    

    def generate_recipe_information(

        self,

        transcript

    ):

        prompt = f"""
You are an expert cooking assistant.

Read the following cooking transcript carefully.

Extract the recipe information.

Return ONLY valid JSON in the following format.

{{
    "dish_type": "Recipe",
    "ingredients": [],
    "estimated_steps": [],
    "cuisine": "",
    "tags": []
}}

Rules:

1. dish_type
- Always return "Recipe".

2. ingredients
- Include only ingredient names.
- Do not include quantities.
- Remove duplicates.

3. estimated_steps
- Write 5 to 10 cooking steps.
- Keep them in chronological order.
- Each step should be one sentence.

4. cuisine
- Predict the cuisine if possible.
- Otherwise return "Unknown".

5. tags
- Return 3 to 6 relevant tags.

Transcript:

{transcript}

Return ONLY JSON.
"""

        try:

            response = ollama.chat(

                model=self.model_name,

                messages=[

                    {

                        "role": "user",

                        "content": prompt

                    }

                ]

            )

            output = response["message"]["content"].strip()

            # Remove markdown if present

            output = output.replace(

                "```json",

                ""

            )

            output = output.replace(

                "```",

                ""

            ).strip()

            recipe_information = json.loads(

                output

            )

            return recipe_information

        except Exception as e:

            print("\nRecipe Information Generation Failed")

            print(e)

            return {

                "dish_type": "",

                "ingredients": [],

                "estimated_steps": [],

                "cuisine": "",

                "tags": []

            }
    
    # Process One Master JSON
    

    def process_json(

        self,

        json_path

    ):

        print(f"\nProcessing {json_path.name}")

        
        # Load Master JSON


        with open(

            json_path,

            "r",

            encoding="utf-8"

        ) as file:

            master_json = json.load(

                file

            )

       
        # Read Audio Information
       

        audio_information = master_json.get(

            "audio_information",

            {}

        )

        transcript = audio_information.get(

            "transcript_en",

            ""

        )

        if transcript.strip() == "":

            print("No English transcript found.")

            return

       
        # Generate Recipe Information
        

        print("Generating Recipe Information...")

        recipe_information = self.generate_recipe_information(

            transcript

        )

        
        # Update Recipe Information
        
        master_json["recipe_information"]["dish_type"] = recipe_information.get(

            "dish_type",

            ""

        )

        master_json["recipe_information"]["ingredients"] = recipe_information.get(

            "ingredients",

            []

        )

        master_json["recipe_information"]["estimated_steps"] = recipe_information.get(

            "estimated_steps",

            []

        )
        master_json["recipe_information"]["cuisine"] = recipe_information.get(

            "cuisine",

            ""

        )

        master_json["recipe_information"]["tags"] = recipe_information.get(

            "tags",

            []

        )
        print(f"Full Path : {json_path.resolve()}")
        print(master_json["recipe_information"])

        
        # Save Updated Master JSON
        
        with open(

            json_path,

            "w",

            encoding="utf-8"

        ) as file:

            json.dump(

                master_json,

                file,

                indent=4,

                ensure_ascii=False

            )

        print(" Recipe Information Updated")

        print(

            f"Ingredients Found : "

            f"{len(master_json['recipe_information']['ingredients'])}"

        )

        print(

            f"Steps Generated : "

            f"{len(master_json['recipe_information']['estimated_steps'])}"

        )
       
    # Process All Master JSON Files
    

    def process_all_json(

        self

    ):

        json_files = self.get_json_files()

        print("\n========================================")
        print("Recipe Information Generation Started")
        print("========================================")

        print(

            f"\nFound {len(json_files)} JSON files.\n"

        )

        for index, json_file in enumerate(

            json_files,

            start=1

        ):

            print(

                f"\n[{index}/{len(json_files)}] "

                f"Processing {json_file.name}"

            )

            self.process_json(

                json_file

            )

        print("\n========================================")
        print(" Recipe Information Generation Completed")
        print("========================================")

# Main

if __name__ == "__main__":

    generator = RecipeInformationGenerator()

    generator.process_all_json()

Loading Gemma 3 Model...
 Model Loaded : gemma3:latest

Recipe Information Generation Started

Found 5 JSON files.


[1/5] Processing CARAMEL_CUSTARD.json

Processing CARAMEL_CUSTARD.json
Generating Recipe Information...
Full Path : /home/debisha/video-graph-rag/data/json/CARAMEL_CUSTARD.json
{'dish_type': 'Recipe', 'ingredients': ['sugar', 'milk', 'eggs', 'egg yolk', 'salt', 'vanilla essence', 'sugar'], 'estimated_steps': ['Boil the milk until it reaches a hot, smoky consistency.', 'Whisk together whole eggs and egg yolk until well combined.', 'Slowly pour in hot milk while continuously stirring the mixture.', 'Add a pinch of salt to balance the flavors and incorporate vanilla essence.', 'Melt sugar in a pan over medium heat with salt.', 'Stir the melted sugar consistently to prevent burning.', 'Poke the sugar with a nail to test for caramelization.'], 'cuisine': 'Unknown', 'tags': ['caramel', 'custard', 'dessert', 'sweet', 'baking']}
 Recipe Information Updated
Ingredients Found : 7


## Step 11: Chunk Generation

### Input
- Master JSON files containing recipe information, frame information, object detection results, and visual descriptions.

### Processing
- Creates a **recipe summary chunk** containing the overall recipe details.
- Generates one **frame-level chunk** for each representative frame by combining the visual description, OCR text, detected objects, and timestamp.
- Organizes all chunks into a structured format suitable for embedding and semantic retrieval.

### Output
- Creates a separate chunk file (`*_chunks.json`) for each video.
- Stores both recipe-level and frame-level chunks with their metadata.
- Saves the generated chunks in the `data/chunks` directory.

In [ ]:

# Import Required Libraries

import json
from pathlib import Path



# Chunk Generator


class ChunkGenerator:

    def __init__(self):

        
        # Master JSON Folder
        
        self.json_directory = Path(

            "data/json"

        )

        
        # Output Chunk Folder
        

        self.chunk_directory = Path(

            "data/chunks"

        )

        # Create folder if it does not exist

        self.chunk_directory.mkdir(

            parents=True,

            exist_ok=True

        )

        print("\n========================================")
        print("Chunk Generator Initialized")
        print("========================================")


    
    # Get All Master JSON Files
    
    def get_json_files(self):

        return sorted(

            self.json_directory.glob(

                "*.json"

            )

        )
    
    # Create Recipe Summary Chunk
    
    def create_recipe_summary_chunk(

        self,

        video_id,

        recipe_information

    ):

        dish = recipe_information.get(

            "dish_type",

            ""

        )

        cuisine = recipe_information.get(

            "cuisine",

            ""

        )

        ingredients = ", ".join(

            recipe_information.get(

                "ingredients",

                []

            )

        )

        steps = " ".join(

            recipe_information.get(

                "estimated_steps",

                []

            )

        )

        text = (

            f"Dish: {dish}. "

            f"Cuisine: {cuisine}. "

            f"Ingredients: {ingredients}. "

            f"Steps: {steps}."

        )

        return {

            "chunk_id": f"{video_id}_summary",

            "video_id": video_id,

            "type": "recipe_summary",

            "timestamp": 0.0,

            "text": text

        }


    
    # Create Frame Chunk
    

    def create_frame_chunk(

        self,

        video_id,

        frame,

        visual_description,

        detected_objects

    ):

        frame_id = frame["frame_id"]

        shot_id = frame["shot_id"]

        timestamp = frame["timestamp"]

        ocr_text = frame.get(

            "ocr_text",

            ""

        )

        object_names = [

            obj["class_name"]

            for obj in detected_objects

        ]

        object_text = ", ".join(

            sorted(

                set(object_names)

            )

        )

        text = (

            f"At {timestamp:.2f}s. "

            f"Visual: {visual_description}. "

            f"OCR: {ocr_text}. "

            f"Objects: {object_text}."

        )

        return {

            "chunk_id":

                f"{video_id}_{shot_id}_{frame_id}",

            "video_id": video_id,

            "type": "frame_chunk",

            "timestamp": timestamp,

            "text": text

        }
    
    # Process One Master JSON
    

    def process_json(

        self,

        json_path

    ):

        print(f"\nProcessing {json_path.name}")

        # ---------------------------------------------------------
        # Load Master JSON
        # ---------------------------------------------------------

        with open(

            json_path,

            "r",

            encoding="utf-8"

        ) as file:

            master_json = json.load(

                file

            )

        # ---------------------------------------------------------
        # Read Required Sections
        # ---------------------------------------------------------

        video_info = master_json.get(

            "video_info",

            {}

        )

        recipe_information = master_json.get(

            "recipe_information",

            {}

        )

        frame_information = master_json.get(

            "frame_information",

            []

        )

        object_detection = master_json.get(

            "object_detection",

            []

        )

        visual_descriptions = master_json.get(

            "visual_descriptions",

            []

        )

        video_id = video_info.get(

            "video_id",

            "UNKNOWN_VIDEO"

        )

        # ---------------------------------------------------------
        # Create Lookup Dictionaries
        # ---------------------------------------------------------

        object_lookup = {

            item["frame_id"]: item["detected_objects"]

            for item in object_detection

        }

        visual_lookup = {

            item["frame_id"]: item["description"]

            for item in visual_descriptions

        }

        # ---------------------------------------------------------
        # Store All Chunks
        # ---------------------------------------------------------

        chunks = []

        # ---------------------------------------------------------
        # Recipe Summary Chunk
        # ---------------------------------------------------------

        recipe_chunk = self.create_recipe_summary_chunk(

            video_id,

            recipe_information

        )

        chunks.append(

            recipe_chunk

        )

        print(" Recipe Summary Chunk Created")
            # ---------------------------------------------------------
        # Generate Frame Chunks
        # ---------------------------------------------------------

        print(

            f"Generating Frame Chunks ({len(frame_information)} Frames)..."

        )

        for frame in frame_information:

            frame_id = frame["frame_id"]

            # ---------------------------------------------------------
            # Get Visual Description
            # ---------------------------------------------------------

            visual_description = visual_lookup.get(

                frame_id,

                ""

            )

            # ---------------------------------------------------------
            # Get Detected Objects
            # ---------------------------------------------------------

            detected_objects = object_lookup.get(

                frame_id,

                []

            )

            # ---------------------------------------------------------
            # Create Frame Chunk
            # ---------------------------------------------------------

            frame_chunk = self.create_frame_chunk(

                video_id,

                frame,

                visual_description,

                detected_objects

            )

            chunks.append(

                frame_chunk

            )

            print(

                f" Frame Chunk Created : {frame_id}"

            )
        # ---------------------------------------------------------
        # Save Chunk File
        # ---------------------------------------------------------

        chunk_file = self.chunk_directory / (

            f"{video_id}_chunks.json"

        )

        with open(

            chunk_file,

            "w",

            encoding="utf-8"

        ) as file:

            json.dump(

                chunks,

                file,

                indent=4,

                ensure_ascii=False

            )

        print(

            f"\n Saved {len(chunks)} Chunks"

        )

        print(

            f"Chunk File : {chunk_file.name}"

        )


    # =========================================================================
    # Process All Master JSON Files
    # =========================================================================

    def process_all_json(

        self

    ):

        json_files = self.get_json_files()

        print("\n========================================")
        print("Chunk Generation Started")
        print("========================================")

        print(

            f"\nFound {len(json_files)} JSON files.\n"

        )

        for index, json_file in enumerate(

            json_files,

            start=1

        ):

            print(

                f"\n[{index}/{len(json_files)}] "

                f"Processing {json_file.name}"

            )

            self.process_json(

                json_file

            )

        print("\n========================================")
        print(" Chunk Generation Completed")
        print("========================================")
# =============================================================================
# Main
# =============================================================================

if __name__ == "__main__":

    generator = ChunkGenerator()

    generator.process_all_json()


Chunk Generator Initialized

Chunk Generation Started

Found 5 JSON files.


[1/5] Processing CARAMEL_CUSTARD.json

Processing CARAMEL_CUSTARD.json
 Recipe Summary Chunk Created
Generating Frame Chunks (62 Frames)...
 Frame Chunk Created : CARAMEL_CUSTARD_shot_1_f1
 Frame Chunk Created : CARAMEL_CUSTARD_shot_2_f1
 Frame Chunk Created : CARAMEL_CUSTARD_shot_3_f1
 Frame Chunk Created : CARAMEL_CUSTARD_shot_4_f1
 Frame Chunk Created : CARAMEL_CUSTARD_shot_5_f1
 Frame Chunk Created : CARAMEL_CUSTARD_shot_6_f1
 Frame Chunk Created : CARAMEL_CUSTARD_shot_7_f1
 Frame Chunk Created : CARAMEL_CUSTARD_shot_8_f1
 Frame Chunk Created : CARAMEL_CUSTARD_shot_9_f1
 Frame Chunk Created : CARAMEL_CUSTARD_shot_10_f1
 Frame Chunk Created : CARAMEL_CUSTARD_shot_11_f1
 Frame Chunk Created : CARAMEL_CUSTARD_shot_12_f1
 Frame Chunk Created : CARAMEL_CUSTARD_shot_13_f1
 Frame Chunk Created : CARAMEL_CUSTARD_shot_14_f1
 Frame Chunk Created : CARAMEL_CUSTARD_shot_15_f1
 Frame Chunk Created : CARAMEL_CUSTARD_sh

# Step 12: Dense Embedding Generation

## Overview

The Dense Embedding Generation module converts the retrieval-ready text chunks into dense vector representations using a pre-trained Sentence Transformer model. These embeddings capture the semantic meaning of each chunk, allowing similar content to be retrieved even when the wording of the query differs from the stored text.

Each generated embedding is stored along with its corresponding metadata, enabling efficient indexing and retrieval in the vector database during the Graph-RAG search process.

The generated embeddings serve as the input for the ChromaDB storage module.

---

## Input

The Dense Embedding Generation module reads the chunk files generated in the previous stage from:

- `data/chunks`

Each chunk contains:

- Chunk ID
- Video ID
- Chunk Type
- Timestamp
- Text Content

---

## Processing

The dense embedding generation process consists of the following steps:

1. Load all chunk JSON files from the `data/chunks` directory.
2. Initialize the pre-trained **Sentence Transformer** model (`all-MiniLM-L6-v2`).
3. Read every chunk individually.
4. Extract the text content from each chunk.
5. Generate a dense embedding vector for the chunk using the Sentence Transformer model.
6. Store the generated embedding together with its metadata, including:
   - Chunk ID
   - Video ID
   - Chunk Type
   - Timestamp
   - Original Text
7. Repeat the process for every chunk in every video.
8. Save the generated embeddings as a new JSON file inside the `data/embeddings` directory.

---

## Output

For every chunk file, the module generates a corresponding embedding file containing semantic vector representations of all chunks.

The generated files are stored as:

```
data/embeddings/
│
├── VIDEO_1_embeddings.json
├── VIDEO_2_embeddings.json
├── VIDEO_3_embeddings.json
└── ...
```

Each embedding entry contains:

- Chunk ID
- Video ID
- Chunk Type
- Timestamp
- Original Text
- Dense Embedding Vector

These embedding files are subsequently used to populate the ChromaDB vector database, enabling semantic similarity search and retrieval within the Graph-RAG pipeline.

In [8]:
# =============================================================================
# Import Required Libraries
# =============================================================================

import json
from pathlib import Path

from sentence_transformers import SentenceTransformer


# =============================================================================
# Dense Embedding Generator
# =============================================================================

class DenseEmbeddingGenerator:

    def __init__(self):

        # Chunk Folder
        self.chunk_directory = Path("data/chunks")

        # Output Folder
        self.embedding_directory = Path("data/embeddings")

        self.embedding_directory.mkdir(
            parents=True,
            exist_ok=True
        )

        print("Loading Sentence Transformer...")

        self.model = SentenceTransformer(
            "all-MiniLM-L6-v2"
        )

        print("Model Loaded")


    # -------------------------------------------------------------------------
    # Get Chunk Files
    # -------------------------------------------------------------------------

    def get_chunk_files(self):

        return sorted(

            self.chunk_directory.glob(
                "*_chunks.json"
            )

        )


    # -------------------------------------------------------------------------
    # Generate Embedding
    # -------------------------------------------------------------------------

    def generate_embedding(self, text):

        embedding = self.model.encode(

            text,

            convert_to_numpy=True

        )

        return embedding.tolist()


    # -------------------------------------------------------------------------
    # Process One Chunk File
    # -------------------------------------------------------------------------

    def process_chunk_file(self, chunk_file):

        print(f"\nProcessing {chunk_file.name}")

        with open(

            chunk_file,

            "r",

            encoding="utf-8"

        ) as file:

            chunks = json.load(file)

        embeddings = []

        for chunk in chunks:

            vector = self.generate_embedding(

                chunk["text"]

            )

            embeddings.append(

                {

                    "chunk_id": chunk["chunk_id"],

                    "video_id": chunk["video_id"],

                    "type": chunk["type"],

                    "timestamp": chunk["timestamp"],

                    "text": chunk["text"],

                    "embedding": vector

                }

            )

            print(

                f"Embedded -> {chunk['chunk_id']}"

            )

        output_file = (

            self.embedding_directory /

            chunk_file.name.replace(

                "_chunks",

                "_embeddings"

            )

        )

        with open(

            output_file,

            "w",

            encoding="utf-8"

        ) as file:

            json.dump(

                embeddings,

                file,

                indent=4,

                ensure_ascii=False

            )

        print(f"Saved -> {output_file.name}")


    # -------------------------------------------------------------------------
    # Process All Chunk Files
    # -------------------------------------------------------------------------

    def process_all_chunks(self):

        chunk_files = self.get_chunk_files()

        print(f"\nFound {len(chunk_files)} chunk files.")

        for chunk_file in chunk_files:

            self.process_chunk_file(

                chunk_file

            )

        print("\nDense Embedding Generation Completed.")


# =============================================================================
# Main
# =============================================================================

if __name__ == "__main__":

    generator = DenseEmbeddingGenerator()

    generator.process_all_chunks()

Loading Sentence Transformer...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model Loaded

Found 5 chunk files.

Processing CARAMEL_CUSTARD_chunks.json
Embedded -> CARAMEL_CUSTARD_summary
Embedded -> CARAMEL_CUSTARD_shot_1_CARAMEL_CUSTARD_shot_1_f1
Embedded -> CARAMEL_CUSTARD_shot_2_CARAMEL_CUSTARD_shot_2_f1
Embedded -> CARAMEL_CUSTARD_shot_3_CARAMEL_CUSTARD_shot_3_f1
Embedded -> CARAMEL_CUSTARD_shot_4_CARAMEL_CUSTARD_shot_4_f1
Embedded -> CARAMEL_CUSTARD_shot_5_CARAMEL_CUSTARD_shot_5_f1
Embedded -> CARAMEL_CUSTARD_shot_6_CARAMEL_CUSTARD_shot_6_f1
Embedded -> CARAMEL_CUSTARD_shot_7_CARAMEL_CUSTARD_shot_7_f1
Embedded -> CARAMEL_CUSTARD_shot_8_CARAMEL_CUSTARD_shot_8_f1
Embedded -> CARAMEL_CUSTARD_shot_9_CARAMEL_CUSTARD_shot_9_f1
Embedded -> CARAMEL_CUSTARD_shot_10_CARAMEL_CUSTARD_shot_10_f1
Embedded -> CARAMEL_CUSTARD_shot_11_CARAMEL_CUSTARD_shot_11_f1
Embedded -> CARAMEL_CUSTARD_shot_12_CARAMEL_CUSTARD_shot_12_f1
Embedded -> CARAMEL_CUSTARD_shot_13_CARAMEL_CUSTARD_shot_13_f1
Embedded -> CARAMEL_CUSTARD_shot_14_CARAMEL_CUSTARD_shot_14_f1
Embedded -> CARAMEL_CUSTA

# Step 13: Sparse Embedding (BM25 Index Generation)

## Overview

The Sparse Embedding module builds a **BM25 index** for all generated text chunks. Unlike dense embeddings, which capture semantic meaning using neural networks, BM25 performs **keyword-based retrieval** by measuring the relevance of documents based on term frequency and inverse document frequency (TF-IDF).

Each chunk is tokenized and indexed using the BM25 algorithm, enabling efficient lexical search based on exact keyword matches. Along with the BM25 index, the corresponding chunk metadata is stored separately to facilitate retrieval during query processing.

This module forms the **sparse retrieval** component of the hybrid retrieval pipeline and complements dense semantic search.

---

## Input

The Sparse Embedding module reads the generated chunk files from:

- `data/chunks`

Each chunk contains:

- Chunk ID
- Video ID
- Chunk Type
- Timestamp
- Text Content

---

## Processing

The sparse embedding generation process consists of the following steps:

1. Load all chunk JSON files from the `data/chunks` directory.
2. Read every chunk and extract its text content.
3. Perform simple preprocessing by:
   - Converting text to lowercase.
   - Splitting the text into tokens.
4. Build a corpus consisting of tokenized chunks.
5. Store the metadata associated with each chunk, including:
   - Chunk ID
   - Video ID
   - Chunk Type
   - Timestamp
   - Original Text
6. Generate a BM25 index using the tokenized corpus.
7. Save the BM25 index as a serialized (`.pkl`) file.
8. Save the corresponding chunk metadata as a JSON file for retrieval during search.

---

## Output

For every chunk file, the module generates two output files:

- A serialized BM25 index.
- A metadata file containing information about every indexed chunk.

The generated files are stored as:

```
data/bm25_index/
│
├── VIDEO_1_bm25.pkl
├── VIDEO_1_metadata.json
├── VIDEO_2_bm25.pkl
├── VIDEO_2_metadata.json
├── VIDEO_3_bm25.pkl
├── VIDEO_3_metadata.json
└── ...
```

The BM25 index enables efficient keyword-based retrieval, while the metadata file allows the retrieved chunk IDs to be mapped back to their original content.

Together with the Dense Embedding Generation module, this forms the **hybrid retrieval** foundation of the Graph-RAG pipeline, combining semantic similarity search with exact keyword matching for improved retrieval performance.

In [ ]:
# =============================================================================
# Import Required Libraries
# =============================================================================

import json
import pickle

from pathlib import Path

from rank_bm25 import BM25Okapi


# =============================================================================
# BM25 Index Generator
# =============================================================================

class BM25IndexGenerator:

    def __init__(self):

        # ---------------------------------------------------------
        # Chunk Folder
        # ---------------------------------------------------------

        self.chunk_directory = Path(

            "data/chunks"

        )

        # ---------------------------------------------------------
        # Output Folder
        # ---------------------------------------------------------

        self.output_directory = Path(

            "data/bm25_index"

        )

        self.output_directory.mkdir(

            parents=True,

            exist_ok=True

        )

        print("\nLoading BM25 Index Generator...")

        print("✓ Ready")


    # =========================================================================
    # Get All Chunk Files
    # =========================================================================

    def get_chunk_files(self):

        return sorted(

            self.chunk_directory.glob(

                "*_chunks.json"

            )

        )
    # =========================================================================
    # Process One Chunk File
    # =========================================================================

    def process_chunk_file(

        self,

        chunk_file

    ):

        print(f"\nProcessing {chunk_file.name}")

        # ---------------------------------------------------------
        # Load Chunk File
        # ---------------------------------------------------------

        with open(

            chunk_file,

            "r",

            encoding="utf-8"

        ) as file:

            chunks = json.load(

                file

            )

        corpus = []

        chunk_metadata = []

        # ---------------------------------------------------------
        # Prepare Corpus
        # ---------------------------------------------------------

        for chunk in chunks:

            text = chunk["text"]

            # Simple tokenization
            tokens = text.lower().split()

            corpus.append(

                tokens

            )

            chunk_metadata.append(

                {

                    "chunk_id": chunk["chunk_id"],

                    "video_id": chunk["video_id"],

                    "type": chunk["type"],

                    "timestamp": chunk["timestamp"],

                    "text": chunk["text"]

                }

            )

        print(

            f"Prepared {len(corpus)} Chunks"

        )

        # ---------------------------------------------------------
        # Build BM25 Index
        # ---------------------------------------------------------

        bm25 = BM25Okapi(

            corpus

        )
    # ---------------------------------------------------------
    # Save BM25 Index
    # ---------------------------------------------------------

        index_file = self.output_directory / (

            chunk_file.name.replace(

                "_chunks.json",

                "_bm25.pkl"

            )

        )

        with open(

            index_file,

            "wb"

        ) as file:

            pickle.dump(

                bm25,

                file

            )

        # ---------------------------------------------------------
        # Save Chunk Metadata
        # ---------------------------------------------------------

        metadata_file = self.output_directory / (

            chunk_file.name.replace(

                "_chunks.json",

                "_metadata.json"

            )

        )

        with open(

            metadata_file,

            "w",

            encoding="utf-8"

        ) as file:

            json.dump(

                chunk_metadata,

                file,

                indent=4,

                ensure_ascii=False

            )

        print(

            f"✓ BM25 Index Saved : {index_file.name}"

        )

        print(

            f"✓ Metadata Saved : {metadata_file.name}"

        )
    # =========================================================================
    # Process All Chunk Files
    # =========================================================================

    def process_all_chunk_files(

        self

    ):

        chunk_files = self.get_chunk_files()

        print("\n========================================")
        print("BM25 Index Generation Started")
        print("========================================")

        print(

            f"\nFound {len(chunk_files)} Chunk Files.\n"

        )

        for index, chunk_file in enumerate(

            chunk_files,

            start=1

        ):

            print(

                f"\n[{index}/{len(chunk_files)}] "

                f"Processing {chunk_file.name}"

            )

            self.process_chunk_file(

                chunk_file

            )

        print("\n========================================")
        print("✓ BM25 Index Generation Completed")
        print("========================================")


# =============================================================================
# Main
# =============================================================================

if __name__ == "__main__":

    generator = BM25IndexGenerator()

    generator.process_all_chunk_files()


Loading BM25 Index Generator...
✓ Ready

BM25 Index Generation Started

Found 5 Chunk Files.


[1/5] Processing CARAMEL_CUSTARD_chunks.json

Processing CARAMEL_CUSTARD_chunks.json
Prepared 63 Chunks
✓ BM25 Index Saved : CARAMEL_CUSTARD_bm25.pkl
✓ Metadata Saved : CARAMEL_CUSTARD_metadata.json

[2/5] Processing DAHI_CHICKEN_chunks.json

Processing DAHI_CHICKEN_chunks.json
Prepared 42 Chunks
✓ BM25 Index Saved : DAHI_CHICKEN_bm25.pkl
✓ Metadata Saved : DAHI_CHICKEN_metadata.json

[3/5] Processing FRIED_RICE_chunks.json

Processing FRIED_RICE_chunks.json
Prepared 34 Chunks
✓ BM25 Index Saved : FRIED_RICE_bm25.pkl
✓ Metadata Saved : FRIED_RICE_metadata.json

[4/5] Processing MANGO_CHICKEN_ROAST_chunks.json

Processing MANGO_CHICKEN_ROAST_chunks.json
Prepared 51 Chunks
✓ BM25 Index Saved : MANGO_CHICKEN_ROAST_bm25.pkl
✓ Metadata Saved : MANGO_CHICKEN_ROAST_metadata.json

[5/5] Processing POMFRET_ROAST_FRY_chunks.json

Processing POMFRET_ROAST_FRY_chunks.json
Prepared 56 Chunks
✓ BM25 Index

# Step 14: ChromaDB Population

## Objective

After generating dense embeddings for every chunk, the next step is to store these embeddings in a vector database. This notebook populates **ChromaDB** with the generated embeddings, enabling efficient semantic similarity search during retrieval.

---

## Input

The notebook reads the dense embedding files generated in the previous step.

**Input Folder**

```
data/dense_embeddings/
```

Each file contains chunk information along with its dense embedding vector.

Example:

```json
{
    "chunk_id": "CARAMEL_CUSTARD_summary",
    "video_id": "CARAMEL_CUSTARD",
    "type": "recipe_summary",
    "timestamp": 0.0,
    "text": "...",
    "embedding": [0.012, -0.143, ...]
}
```

---

## Processing

The ChromaDB population process consists of the following steps:

1. Load all dense embedding JSON files.
2. Initialize a persistent ChromaDB client.
3. Create (or load) the `video_rag` collection.
4. Read every embedded chunk.
5. Insert each chunk into ChromaDB.
6. Store:
   - Chunk ID
   - Embedding Vector
   - Chunk Text
   - Metadata (video ID, chunk type, timestamp)
7. Repeat until all embedding files have been indexed.

---

## Output

A persistent ChromaDB vector database is created.

**Output Folder**

```
data/chroma_db/
```

The database contains one collection:

```
video_rag
```

Each stored entry contains:

- Chunk ID
- Dense Embedding Vector
- Chunk Text
- Video ID
- Chunk Type
- Timestamp

This database is later queried by the Hybrid Retriever for dense semantic search.

---

## Summary

This step builds the **vector database** for the Text-RAG pipeline. Instead of scanning every chunk during retrieval, ChromaDB indexes all dense embeddings and performs efficient similarity search, allowing the system to quickly retrieve the most semantically relevant recipe chunks for a user query.

In [10]:
# =============================================================================
# Import Required Libraries
# =============================================================================

import json

from pathlib import Path

import chromadb


# =============================================================================
# ChromaDB Generator
# =============================================================================

class ChromaDBGenerator:

    def __init__(self):

        # ---------------------------------------------------------
        # Dense Embedding Folder
        # ---------------------------------------------------------

        self.embedding_directory = Path(

            "data/dense_embeddings"

        )

        # ---------------------------------------------------------
        # Chroma Database Folder
        # ---------------------------------------------------------

        self.database_directory = "data/chroma_db"

        # ---------------------------------------------------------
        # Create Persistent Client
        # ---------------------------------------------------------

        self.client = chromadb.PersistentClient(

            path=self.database_directory

        )

        # ---------------------------------------------------------
        # Create Collection
        # ---------------------------------------------------------

        self.collection = self.client.get_or_create_collection(

            name="video_rag"

        )

        print("\n========================================")
        print("ChromaDB Initialized")
        print("========================================")


    # =========================================================================
    # Get Dense Embedding Files
    # =========================================================================

    def get_dense_files(self):

        return sorted(

            self.embedding_directory.glob(

                "*_dense.json"

            )

        )
    # =========================================================================
    # Process One Dense Embedding File
    # =========================================================================

    def process_dense_file(

        self,

        dense_file

    ):

        print(f"\nProcessing {dense_file.name}")

        # ---------------------------------------------------------
        # Load Dense Embeddings
        # ---------------------------------------------------------

        with open(

            dense_file,

            "r",

            encoding="utf-8"

        ) as file:

            embedded_chunks = json.load(

                file

            )

        # ---------------------------------------------------------
        # Insert Each Chunk into ChromaDB
        # ---------------------------------------------------------

        for index, chunk in enumerate(

            embedded_chunks,

            start=1

        ):

            print(

                f"Inserting Chunk {index}/{len(embedded_chunks)}"

            )

            self.collection.add(

                ids=[

                    chunk["chunk_id"]

                ],

                embeddings=[

                    chunk["embedding"]

                ],

                documents=[

                    chunk["text"]

                ],

                metadatas=[

                    {

                        "video_id": chunk["video_id"],

                        "type": chunk["type"],

                        "timestamp": chunk["timestamp"]

                    }

                ]

            )

        print(

            f"✓ Inserted {len(embedded_chunks)} Chunks"

        )
    # =========================================================================
    # Process All Dense Embedding Files
    # =========================================================================

    def process_all_dense_files(

        self

    ):

        dense_files = self.get_dense_files()

        print("\n========================================")
        print("ChromaDB Population Started")
        print("========================================")

        print(

            f"\nFound {len(dense_files)} Dense Embedding Files.\n"

        )

        for index, dense_file in enumerate(

            dense_files,

            start=1

        ):

            print(

                f"\n[{index}/{len(dense_files)}] "

                f"Processing {dense_file.name}"

            )

            self.process_dense_file(

                dense_file

            )

        print("\n========================================")
        print("✓ ChromaDB Population Completed")
        print("========================================")


# =============================================================================
# Main
# =============================================================================

if __name__ == "__main__":

    generator = ChromaDBGenerator()

    generator.process_all_dense_files()


ChromaDB Initialized

ChromaDB Population Started

Found 5 Dense Embedding Files.


[1/5] Processing CARAMEL_CUSTARD_dense.json

Processing CARAMEL_CUSTARD_dense.json
Inserting Chunk 1/63
Inserting Chunk 2/63
Inserting Chunk 3/63
Inserting Chunk 4/63
Inserting Chunk 5/63
Inserting Chunk 6/63
Inserting Chunk 7/63
Inserting Chunk 8/63
Inserting Chunk 9/63
Inserting Chunk 10/63
Inserting Chunk 11/63
Inserting Chunk 12/63
Inserting Chunk 13/63
Inserting Chunk 14/63
Inserting Chunk 15/63
Inserting Chunk 16/63
Inserting Chunk 17/63
Inserting Chunk 18/63
Inserting Chunk 19/63
Inserting Chunk 20/63
Inserting Chunk 21/63
Inserting Chunk 22/63
Inserting Chunk 23/63
Inserting Chunk 24/63
Inserting Chunk 25/63
Inserting Chunk 26/63
Inserting Chunk 27/63
Inserting Chunk 28/63
Inserting Chunk 29/63
Inserting Chunk 30/63
Inserting Chunk 31/63
Inserting Chunk 32/63
Inserting Chunk 33/63
Inserting Chunk 34/63
Inserting Chunk 35/63
Inserting Chunk 36/63
Inserting Chunk 37/63
Inserting Chunk 38/63
Inser

# Step 15: Hybrid Retrieval

## Description

This step retrieves the most relevant chunks for a user query using a **Hybrid Retrieval** approach. It combines **Dense Retrieval (semantic search)** using ChromaDB with **Sparse Retrieval (keyword search)** using BM25. The retrieved results are merged to improve retrieval accuracy.

---

## Input

- ChromaDB (`data/chroma_db/`)
- BM25 Index (`data/bm25_index/`)
- SentenceTransformer (`all-MiniLM-L6-v2`)
- User Query

---

## Processing

- Load the ChromaDB collection, BM25 indexes, and SentenceTransformer model.
- Convert the user query into a dense embedding.
- Perform **Dense Retrieval** by searching the query embedding in ChromaDB.
- Perform **Sparse Retrieval** by searching the tokenized query using BM25.
- Merge dense and sparse results.
- Label each retrieved chunk as **Dense**, **Sparse**, or **Hybrid**.
- Display the retrieved chunks with their metadata and scores.

> **Note:** During dense retrieval, ChromaDB compares the query embedding with stored chunk embeddings using its configured **distance metric**. Since the collection is created without specifying `hnsw:space`, ChromaDB uses its default distance metric (typically **L2/Euclidean distance**). Therefore, the returned values are **vector distances**, not cosine similarity scores. A **smaller distance indicates a better semantic match**.

---

## Output

The retriever returns the most relevant chunks containing:

- Chunk ID
- Video ID
- Chunk Type
- Timestamp
- Retrieval Source (Dense / Sparse / Hybrid)
- Retrieval Score
- Chunk Text

---

## Scoring

### Dense Retrieval (ChromaDB)

- Score = **Vector Distance**
- Obtained from `results["distances"]`
- **Lower distance = Better semantic similarity**

### Sparse Retrieval (BM25)

- Score = **BM25 Relevance Score**
- Obtained using `bm25.get_scores()`
- **Higher score = Better keyword match**

The current implementation does **not** combine these scores into a single ranking score. Instead, it merges the results returned by both retrieval methods.

In [1]:
# =============================================================================
# Import Required Libraries
# =============================================================================

import json
import pickle

from pathlib import Path

import chromadb
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
# =============================================================================
# Hybrid Retriever
# =============================================================================

class HybridRetriever:

    def __init__(self):

        # ChromaDB
        self.client = chromadb.PersistentClient(
            path="data/chroma_db"
        )

        self.collection = self.client.get_collection(
            "video_rag"
        )

        # BM25 Folder
        self.bm25_directory = Path(
            "data/bm25_index"
        )

        # Dense Embedding Model
        print("Loading SentenceTransformer...")

        self.embedding_model = SentenceTransformer(
            "all-MiniLM-L6-v2"
        )

        print("✓ Model Loaded")
# =============================================================================
# Generate Query Embedding
# =============================================================================

def generate_query_embedding(
    self,
    query
):

    embedding = self.embedding_model.encode(
        query,
        convert_to_numpy=True
    )

    return embedding.tolist()

HybridRetriever.generate_query_embedding = generate_query_embedding
# =============================================================================
# Dense Search
# =============================================================================

def dense_search(
    self,
    query,
    top_k=5
):

    print("\nPerforming Dense Search...")

    query_embedding = self.generate_query_embedding(
        query
    )

    results = self.collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    dense_results = []

    ids = results["ids"][0]
    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    for i in range(len(ids)):

        dense_results.append(
            {
                "chunk_id": ids[i],
                "text": documents[i],
                "video_id": metadatas[i]["video_id"],
                "type": metadatas[i]["type"],
                "timestamp": metadatas[i]["timestamp"],
                "score": distances[i]
            }
        )

    print(f"Retrieved {len(dense_results)} Dense Results")

    return dense_results

HybridRetriever.dense_search = dense_search
# =============================================================================
# Sparse Search
# =============================================================================

def sparse_search(
    self,
    query,
    top_k=5
):

    print("\nPerforming Sparse Search...")

    sparse_results = []

    query_tokens = query.lower().split()

    bm25_files = sorted(
        self.bm25_directory.glob("*_bm25.pkl")
    )

    for bm25_file in bm25_files:

        with open(bm25_file, "rb") as file:
            bm25 = pickle.load(file)

        metadata_file = bm25_file.with_name(
            bm25_file.name.replace(
                "_bm25.pkl",
                "_metadata.json"
            )
        )

        with open(metadata_file, "r", encoding="utf-8") as file:
            metadata = json.load(file)

        scores = bm25.get_scores(query_tokens)

        top_indices = sorted(
            range(len(scores)),
            key=lambda i: scores[i],
            reverse=True
        )[:top_k]

        for index in top_indices:

            sparse_results.append(
                {
                    "chunk_id": metadata[index]["chunk_id"],
                    "text": metadata[index]["text"],
                    "video_id": metadata[index]["video_id"],
                    "type": metadata[index]["type"],
                    "timestamp": metadata[index]["timestamp"],
                    "score": float(scores[index])
                }
            )

    sparse_results = sorted(
        sparse_results,
        key=lambda x: x["score"],
        reverse=True
    )

    print(f"Retrieved {len(sparse_results[:top_k])} Sparse Results")

    return sparse_results[:top_k]

HybridRetriever.sparse_search = sparse_search
# =============================================================================
# Hybrid Search
# =============================================================================

def hybrid_search(
    self,
    query,
    dense_k=5,
    sparse_k=5
):

    dense_results = self.dense_search(query, dense_k)

    sparse_results = self.sparse_search(query, sparse_k)

    hybrid_results = {}

    for result in dense_results:

        hybrid_results[result["chunk_id"]] = result
        hybrid_results[result["chunk_id"]]["retrieval"] = "Dense"

    for result in sparse_results:

        chunk_id = result["chunk_id"]

        if chunk_id in hybrid_results:
            hybrid_results[chunk_id]["retrieval"] = "Hybrid"
        else:
            hybrid_results[chunk_id] = result
            hybrid_results[chunk_id]["retrieval"] = "Sparse"

    final_results = list(hybrid_results.values())

    print(f"\nTotal Hybrid Results : {len(final_results)}")

    return final_results

HybridRetriever.hybrid_search = hybrid_search
# =============================================================================
# Process Query
# =============================================================================

def process_query(
    self,
    query
):

    print("\n========================================")
    print("Hybrid Retrieval Started")
    print("========================================")

    print(f"\nQuery : {query}")

    results = self.hybrid_search(query)

    print("\n========================================")
    print("Retrieved Chunks")
    print("========================================")

    for index, result in enumerate(results, start=1):

        print(f"\nRank : {index}")
        print(f"Chunk ID  : {result['chunk_id']}")
        print(f"Video ID  : {result['video_id']}")
        print(f"Type      : {result['type']}")
        print(f"Timestamp : {result['timestamp']}")
        print(f"Source    : {result['retrieval']}")
        print(f"Score     : {result['score']}")
        print(result["text"][:250])

    return results

HybridRetriever.process_query = process_query
# =============================================================================
# Main Execution
# =============================================================================

# Create Hybrid Retriever
retriever = HybridRetriever()

# Example Query
query = "when is milk added in caramel custard"

# Perform Hybrid Retrieval
results = retriever.process_query(
    query
)

Loading SentenceTransformer...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✓ Model Loaded

Hybrid Retrieval Started

Query : when is milk added in caramel custard

Performing Dense Search...
Retrieved 5 Dense Results

Performing Sparse Search...
Retrieved 5 Sparse Results

Total Hybrid Results : 10

Retrieved Chunks

Rank : 1
Chunk ID  : CARAMEL_CUSTARD_shot_28_CARAMEL_CUSTARD_shot_28_f1
Video ID  : CARAMEL_CUSTARD
Type      : frame_chunk
Timestamp : 39.3
Source    : Dense
Score     : 0.38293084502220154
Recipe: Caramel Custard. Timestamp: 39.30 seconds. Cooking scene: In this creamy recipe, gently stir the mixture as milk is slowly added while it’s being whipped to achieve a smooth and airy consistency. Keep stirring until the desired texture is rea

Rank : 2
Chunk ID  : CARAMEL_CUSTARD_shot_1_CARAMEL_CUSTARD_shot_1_f1
Video ID  : CARAMEL_CUSTARD
Type      : frame_chunk
Timestamp : 0.65
Source    : Dense
Score     : 0.4142149090766907
Recipe: Caramel Custard. Timestamp: 0.65 seconds. Cooking scene: A person is opening a package of "Cow Milk" and pouring it i